# a08 오른쪽 무릎과 발목 · 두 시점 나란히 판독
**23번 무릎 통과**와 **29번 drive(바와 몸의 접촉)**에서 선수 본인의 오른쪽 무릎 관절선과 복사뼈를 같은 프레임에서 직접 짚을 수 있는지 확인합니다. 프레임 번호는 원본의 첫 장을 0으로 셉니다.

**모든 셀 실행**을 누르고 두 장을 나란히 보세요. 각 사진은 따로 저장되며 중단 후 다시 열어도 이어서 판독할 수 있습니다.

### 1. 원본과 시점 확인

In [ ]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from wlcv.annotation.pose_points import load_points, pose_point_editor, pose_point_path
from wlcv.annotation.storage import load_events
from wlcv.annotation.train_pose import STATUS_LABELS
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS

video_id = "a08_sn_f_01"
info = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id").loc[video_id]
split = pd.read_csv(SPLIT).set_index("video_id").loc[video_id]
if (info.person_id, split.person_id, split.role, info.near_side) != ("a08", "a08", "train", "right"):
    raise ValueError("a08의 학습 분할과 가까운 쪽을 확인하세요.")

scenes = [(23, "무릎 통과"), (29, "drive")]
for number in (1, 2):
    events = load_events(ANNOTATIONS, video_id, number)
    if (events.get("transition"), events.get("drive")) != (23, 29):
        raise ValueError(f"{number}차 이벤트 판독에서 두 프레임을 확인하세요.")
joints = ("무릎", "발목")
folder = ANNOTATIONS / "a08_right_knee_ankle"

### 2. 같은 쪽의 두 관절 표시하기
각 사진에서 **무릎** 또는 **발목**을 선택하세요. 무릎은 **관절선**, 발목은 **복사뼈**를 기준으로 합니다. 윤곽에서 기준점을 직접 구분할 수 있으면 **직접 확인**을 선택하고 클릭하세요. 위치를 짐작할 수 있으면 **추정 사유**를 고른 뒤 클릭하세요. 좌표를 정하기 어려우면 **가려서 판독 불가** 또는 **위치 불확실**을 누르세요.

십자는 직접 확인, 마름모는 추정입니다. 관절을 다시 클릭하면 해당 점만 수정됩니다. **원본 크기로 확대**를 선택하면 작은 관절을 더 크게 볼 수 있습니다.

In [ ]:
panels = []
for frame, phase in scenes:
    editor = pose_point_editor(
        VIDEOS / info.filename, frame, info.person_id, info.near_side,
        split=split.role, annotations=folder, joints=joints, trunk_review=True,
        frame_label=f"a08 · {frame}번 · {phase}",
    )
    editor.layout.min_width = "0"
    panels.append(editor)

layout = widgets.Layout(grid_template_columns="repeat(2, minmax(330px, 1fr))",
                        grid_gap="12px", width="100%", overflow="auto")
display(widgets.GridBox(children=panels, layout=layout))

### 3. 두 점을 함께 확인했는지 보기
점 네 개의 판독을 마치고 **저장 결과 확인**을 누르세요. 한 프레임에서 무릎과 발목이 모두 직접 확인일 때만 **예**로 셉니다. 판독 불가는 좌표가 비어 있는 완료 판독입니다. CSV는 `data/annotations/a08_right_knee_ankle/personal_pose_points/`에 저장됩니다.

In [ ]:
def show_saved(_=None):
    rows = []
    count = 0
    for frame, phase in scenes:
        points = load_points(pose_point_path(folder, video_id, frame))
        status = [points.get(joint, {}).get("status") for joint in joints]
        count += sum(value is not None for value in status)
        together = "미검수 항목 있음"
        if all(status):
            together = "예" if status == ["located", "located"] else "아니오"
        rows.append({"프레임": frame, "시점": phase,
                     "무릎": STATUS_LABELS.get(status[0], "미검수"),
                     "발목": STATUS_LABELS.get(status[1], "미검수"),
                     "두 관절 모두 직접 확인": together})
    result.clear_output(wait=True)
    with result:
        print(f"{count}/4개 판독 저장")
        display(pd.DataFrame(rows).set_index("프레임"))

refresh = widgets.Button(description="저장 결과 확인")
result = widgets.Output()
refresh.on_click(show_saved)
display(widgets.VBox([refresh, result]))
show_saved()